# Revision R2.2c: GPT-5.4 few-shot condition

**Reviewer 2, comment 2(c):** *"The GPT-5.4 baseline uses a single zero-shot prompt, so please add a few-shot condition or restrict the claim that fine-tuned models 'consistently and substantially' outperform LLM prompting."*

**What this notebook does**
* **5-shot × 3 seeds** (what the reviewer asked for): the zero-shot system prompt from `gpt_zero_shot.ipynb`, unchanged, plus 5 annotated **training** queries as example turns, covering all 4 entity types. Each seed (42, 123, 456) uses a different set of 5 examples, which shows how much the result depends on the choice of examples.
* Everything else is identical to the zero-shot run: query sent as space-joined tokens, `[{"token","label"}]` output, temperature 0, max_completion_tokens 4096, 5 retries, same token-alignment recovery. The zero-shot result already in the paper (F1 = 0.6682) is kept as is.

**Runs:** 3 × 275 queries = 825 API calls, about **$15–20**. Responses are cached on Drive, so it resumes after interruptions without paying twice. Check the cost on the OpenAI dashboard after the first ~25 queries.

**API key:** add it as a Colab **secret** named `OPENAI_API_KEY` (key icon, left sidebar). Never paste it into a cell.

**Runtime:** no GPU needed. Choose **CPU** (*Runtime → Change runtime type → CPU*) to save compute units.

**How to run:** set the runtime (see below) and click **Run all**. Results go to `MyDrive/strokeid_revision/runs`, the same folder for every notebook. Finished runs are skipped, so after a disconnect just **Run all** again. Notebooks can share runs: for example, M3 trained in one notebook is reused by the others.

In [ ]:
!pip install -q pytorch-crf seqeval openai

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
PROJECT = '/content/drive/MyDrive/strokeid_revision'   # <- same folder in every notebook
DATA, RUNS, RESULTS, CODE = f'{PROJECT}/data', f'{PROJECT}/runs', f'{PROJECT}/results', '/content/code'
for d in (RUNS, RESULTS, f'{CODE}/prompts'): os.makedirs(d, exist_ok=True)
sys.path.insert(0, CODE)
for f in ('train.tsv', 'val.tsv', 'test.tsv'):
    assert os.path.exists(f'{DATA}/{f}'), f'missing {DATA}/{f} (copies of the *_fixed.tsv files)'
print('setup OK')

In [ ]:
%%writefile /content/code/common.py
"""Shared helpers for the StrokeID-NER revision experiments.

Data format (one of):
  * CoNLL/TSV: one "token<TAB>label" per line, blank line between queries.
  * JSONL: one {"id": ..., "tokens": [...], "labels": [...]} object per line.
"""
import json
import os
from collections import Counter

ENTITY_TYPES = ["DGN", "RF", "SYM", "TMP"]
LABELS = ["O", "B-SYM", "I-SYM", "B-DGN", "I-DGN", "B-TMP", "I-TMP", "B-RF", "I-RF"]


def load_split(path):
    """Return a list of dicts: {id, tokens, labels}."""
    if path.endswith(".jsonl"):
        out = []
        with open(path, encoding="utf-8") as f:
            for i, line in enumerate(f):
                if line.strip():
                    d = json.loads(line)
                    d.setdefault("id", i)
                    out.append(d)
        return out
    out, toks, labs = [], [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line.strip():
                if toks:
                    out.append({"id": len(out), "tokens": toks, "labels": labs})
                    toks, labs = [], []
                continue
            parts = line.split("\t") if "\t" in line else line.split()
            toks.append(parts[0])
            labs.append(parts[-1])
    if toks:
        out.append({"id": len(out), "tokens": toks, "labels": labs})
    return out


def find_split(data_dir, name):
    for ext in (".jsonl", ".tsv", ".conll", ".txt"):
        p = os.path.join(data_dir, name + ext)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"No {name}.(jsonl|tsv|conll|txt) in {data_dir}")


def get_spans(labels):
    """Extract (type, start, end_inclusive) spans from a BIO sequence.

    Mirrors seqeval's default (non-strict) behaviour: an I- tag that does not
    continue a span of the same type starts a new span.
    """
    spans, cur = [], None
    for i, lab in enumerate(list(labels) + ["O"]):
        tag, typ = (lab.split("-", 1) + [None])[:2] if lab != "O" else ("O", None)
        if cur is not None and (tag in ("O", "B") or typ != cur[0]):
            spans.append((cur[0], cur[1], i - 1))
            cur = None
        if tag == "B" or (tag == "I" and cur is None):
            cur = (typ, i)
    return spans


def per_query_counts(gold_seqs, pred_seqs, types=ENTITY_TYPES):
    """Array-like counts [n_queries][type] -> (tp, n_pred, n_gold), exact match."""
    rows = []
    for g, p in zip(gold_seqs, pred_seqs):
        gs, ps = set(get_spans(g)), set(get_spans(p))
        row = []
        for t in types:
            gt = {s for s in gs if s[0] == t}
            pt = {s for s in ps if s[0] == t}
            row.append((len(gt & pt), len(pt), len(gt)))
        rows.append(row)
    return rows


def macro_from_totals(tp, npred, ngold):
    """tp/npred/ngold are per-type sequences. Returns macro P, R, F1 and per-type F1
    (seqeval 'macro avg' definition: unweighted mean of per-type scores)."""
    ps, rs, fs = [], [], []
    for a, b, c in zip(tp, npred, ngold):
        p = a / b if b else 0.0
        r = a / c if c else 0.0
        f = 2 * p * r / (p + r) if (p + r) else 0.0
        ps.append(p); rs.append(r); fs.append(f)
    n = len(fs)
    return sum(ps) / n, sum(rs) / n, sum(fs) / n, fs


def write_predictions(path, examples, preds):
    with open(path, "w", encoding="utf-8") as f:
        for ex, p in zip(examples, preds):
            assert len(p) == len(ex["tokens"]), (ex["id"], len(p), len(ex["tokens"]))
            f.write(json.dumps({"id": ex["id"], "tokens": ex["tokens"],
                                "gold": ex["labels"], "pred": p}, ensure_ascii=False) + "\n")


def read_predictions(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


def label_stats(examples):
    c = Counter()
    for ex in examples:
        for t, *_ in get_spans(ex["labels"]):
            c[t] += 1
    return dict(c)


In [ ]:
%%writefile /content/code/gpt_fewshot.py
"""GPT-5.4 zero-shot and few-shot conditions (R2.2c).

Reproduces gpt_zero_shot.ipynb exactly for k=0 (same system prompt, the query
sent as space-joined tokens, [{"token","label"}] output, temperature=0,
max_completion_tokens=4096, up to 5 retries, the same token-alignment recovery,
all-O after 5 failures). For k>0 the same k annotated TRAINING queries are added
as user/assistant demonstration turns before every test query.

Demonstrations: chosen once per --seed from the training split, greedily so all
four entity types appear, then filled up to k. Different seeds = different
demonstration sets (gives SD across demonstration choice).

Usage
  export OPENAI_API_KEY=...
  python gpt_fewshot.py --data_dir data --k 0 --seed 42 --out_dir runs/gpt0
  python gpt_fewshot.py --data_dir data --k 5 --seed 42 --out_dir runs/gpt5
Responses are cached in raw_responses.jsonl so an interrupted run resumes.
"""
import argparse
import json
import os
import random
import time

from common import ENTITY_TYPES, LABELS, find_split, get_spans, load_split, write_predictions

HERE = os.path.dirname(os.path.abspath(__file__))


def align_gpt_to_input(input_tokens, gpt_items):
    """
    Align GPT token-label pairs back to the original input token list.

    GPT sometimes merges multiple input tokens into one item (e.g. 'tidak bisa bicara'),
    splits one input token into multiple items (e.g. 'tiba-tiba' -> ['tiba','-','tiba']),
    or skips punctuation. This function recovers correct labels for all input tokens
    even when counts don't match, instead of discarding all labels.
    """
    result  = ['O'] * len(input_tokens)
    inp_idx = 0

    def try_merge_from(pos):
        """Check if gpt_tok matches a merge of input tokens starting at pos."""
        consumed = ''
        j = pos
        while j < len(input_tokens) and len(consumed) <= len(gpt_tok) + 5:
            consumed = (consumed + ' ' + input_tokens[j]).strip()
            if consumed.lower() == gpt_tok.lower():
                return j
            if len(consumed) > len(gpt_tok) + 2:
                break
            j += 1
        return None

    for item in gpt_items:
        if inp_idx >= len(input_tokens):
            break
        gpt_tok = item.get('token', '').strip()
        gpt_lab = item.get('label', 'O')
        if gpt_lab not in LABELS:
            gpt_lab = 'O'
        if not gpt_tok:
            continue

        def assign_merge(start, end, lab):
            """Assign lab to start, and the continuation tag to start+1..end."""
            for k in range(start, end + 1):
                if k == start:
                    result[k] = lab
                else:
                    result[k] = ('I-' + lab[2:]) if lab.startswith('B-') else lab

        #Try at current position
        if gpt_tok.lower() == input_tokens[inp_idx].lower():
            result[inp_idx] = gpt_lab
            inp_idx += 1
            continue

        merge_end = try_merge_from(inp_idx)
        if merge_end is not None:
            assign_merge(inp_idx, merge_end, gpt_lab)
            inp_idx = merge_end + 1
            continue

        if (len(gpt_tok) < len(input_tokens[inp_idx]) and
                gpt_tok.lower() in input_tokens[inp_idx].lower()):
            result[inp_idx] = gpt_lab
            inp_idx += 1
            continue

        #GPT skipped input tokens — look ahead up to 3 positions ─
        for lookahead in range(1, 4):
            ahead = inp_idx + lookahead
            if ahead >= len(input_tokens):
                break
            if gpt_tok.lower() == input_tokens[ahead].lower():
                inp_idx = ahead
                result[inp_idx] = gpt_lab
                inp_idx += 1
                break
            merge_end = try_merge_from(ahead)
            if merge_end is not None:
                inp_idx = ahead
                assign_merge(inp_idx, merge_end, gpt_lab)
                inp_idx = merge_end + 1
                break
            if (len(gpt_tok) < len(input_tokens[ahead]) and
                    gpt_tok.lower() in input_tokens[ahead].lower()):
                inp_idx = ahead
                result[inp_idx] = gpt_lab
                inp_idx += 1
                break
        # If still unmatched: GPT hallucinated a token — skip it silently

    return result




def pick_demos(train, k, seed):
    rng = random.Random(seed)
    pool = [q for q in train if 5 <= len(q["tokens"]) <= 60]
    rng.shuffle(pool)
    chosen, covered = [], set()
    for q in pool:                                   # first cover every entity type
        types = {t for t, *_ in get_spans(q["labels"])}
        if types - covered:
            chosen.append(q); covered |= types
        if covered >= set(ENTITY_TYPES) or len(chosen) == k:
            break
    for q in pool:                                   # then fill up to k
        if len(chosen) >= k:
            break
        if q not in chosen:
            chosen.append(q)
    return chosen[:k]


def answer_json(q):
    return json.dumps([{"token": t, "label": l} for t, l in zip(q["tokens"], q["labels"])], ensure_ascii=False)


def build_messages(system_prompt, demos, q):
    msgs = [{"role": "system", "content": system_prompt}]
    for d in demos:
        msgs.append({"role": "user", "content": " ".join(d["tokens"])})
        msgs.append({"role": "assistant", "content": answer_json(d)})
    msgs.append({"role": "user", "content": " ".join(q["tokens"])})
    return msgs


def parse(raw, tokens):
    """Same logic as the notebook. Returns (labels, status) or raises on bad JSON."""
    raw = raw.strip().replace("```json", "").replace("```", "").strip()
    result = json.loads(raw)
    if len(result) == len(tokens):
        return [l if l in LABELS else "O" for l in (item.get("label", "O") for item in result)], "exact"
    return align_gpt_to_input(tokens, result), "aligned"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", required=True)
    ap.add_argument("--system_prompt", default=os.path.join(HERE, "prompts", "zero_shot_system.txt"))
    ap.add_argument("--model", default="gpt-5.4")
    ap.add_argument("--k", type=int, default=5)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--max_retries", type=int, default=5)
    a = ap.parse_args()
    from openai import OpenAI
    client = OpenAI()

    out = os.path.join(a.out_dir, f"seed{a.seed}"); os.makedirs(out, exist_ok=True)
    train = load_split(find_split(a.data_dir, "train"))
    test = load_split(find_split(a.data_dir, "test"))
    demos = pick_demos(train, a.k, a.seed) if a.k else []
    json.dump([{"id": d["id"], "tokens": d["tokens"], "labels": d["labels"]} for d in demos],
              open(os.path.join(out, "demonstrations.json"), "w"), indent=1, ensure_ascii=False)
    system_prompt = open(a.system_prompt, encoding="utf-8").read()

    cache_p = os.path.join(out, "raw_responses.jsonl")
    cache = {}
    if os.path.exists(cache_p):
        for l in open(cache_p, encoding="utf-8"):
            r = json.loads(l); cache[str(r["id"])] = r
    preds, status = [], {"exact": 0, "aligned": 0, "failed_all_O": 0}
    with open(cache_p, "a", encoding="utf-8") as cf:
        for n, q in enumerate(test):
            rec = cache.get(str(q["id"]))
            if rec is None:
                labs, st, raws, wait = None, "failed_all_O", [], 2
                for attempt in range(a.max_retries):
                    try:
                        r = client.chat.completions.create(
                            model=a.model, messages=build_messages(system_prompt, demos, q),
                            temperature=0, max_completion_tokens=4096)
                        raw = r.choices[0].message.content or ""
                        raws.append(raw)
                        labs, st = parse(raw, q["tokens"])
                        break
                    except Exception as e:
                        err = str(e)
                        if "rate_limit" in err.lower() or "429" in err:
                            print(f"  rate limit, waiting {wait}s"); time.sleep(wait); wait = min(wait * 2, 60)
                        elif attempt < a.max_retries - 1:
                            time.sleep(wait)
                        else:
                            print(f"  query {q['id']}: failed after {a.max_retries} attempts: {err[:80]}")
                if labs is None:
                    labs = ["O"] * len(q["tokens"])
                rec = {"id": q["id"], "labels": labs, "status": st, "raw": raws}
                cf.write(json.dumps(rec, ensure_ascii=False) + "\n"); cf.flush()
                time.sleep(0.5)
            preds.append(rec["labels"]); status[rec["status"]] += 1
            if (n + 1) % 25 == 0:
                print(f"  {n+1}/{len(test)}")
    write_predictions(os.path.join(out, "test_predictions.jsonl"), test, preds)
    json.dump(dict(model=a.model, k=a.k, seed=a.seed, temperature=0, max_completion_tokens=4096,
                   parse_status=status, demo_ids=[d["id"] for d in demos]),
              open(os.path.join(out, "metrics.json"), "w"), indent=2)
    print("parse status:", status, "-> report these counts in the paper")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/code/prompts/zero_shot_system.txt
You are a clinical Named Entity Recognition (NER) system for Indonesian stroke patient queries.

Label each token with one of these BIO tags:
  B-SYM / I-SYM : Symptoms or functional deficits reported as a consequence of stroke
                  (e.g., lemas, kesemutan, tidak bisa bicara, mulut miring, afasia)
  B-DGN / I-DGN : Stroke diagnosis or lay description of stroke pathology
                  (e.g., stroke, stroke ringan, pendarahan di otak, penyumbatan di otak)
  B-TMP / I-TMP : Temporal expressions — when symptoms started or how long they lasted
                  (e.g., tiba-tiba, sudah 3 hari, sejak kemarin, mendadak)
  B-RF  / I-RF  : Risk factors — pre-existing conditions, age, lifestyle, or trauma
                  (e.g., hipertensi, diabetes, usia 65 tahun, kolesterol, kecelakaan)
  O             : Not an entity

Key annotation rules:
1. Stroke spelling variants are ALL B-DGN: struk, strok, storke, stoke, setruk, sroke
2. Age is B-RF and must include the full phrase: 'usia 65 tahun' — all three tokens are RF
3. Negated symptoms (e.g., 'tidak bisa bicara') — SYM (functional deficit even if negated)
4. Negated risk factors (e.g., 'tidak ada hipertensi') — O (condition is absent)
5. Numeric blood pressure values (190/100) — O; label the condition name (hipertensi) as RF
6. Body parts alone (tangan, kaki) — O; only label clinical expressions (mulut miring) as SYM
7. Recovery descriptions (sudah membaik, tidak kunjung sembuh) — O, not SYM
8. Medical procedures (CT scan, MRI, USG) — O
9. 'bangun tidur' (woke from sleep) — O, not TMP
10. 'sudah' + duration (sudah 3 hari) — TMP; 'sudah' + recovery description — O

CRITICAL output requirement:
Return ONLY a JSON array with exactly one object per input token, in the same order:
[{"token": "...", "label": "..."}, ...]
The number of objects MUST equal the number of input tokens.
Do NOT merge tokens. Do NOT split tokens. Do NOT skip tokens.
No markdown, no explanation — only the raw JSON array.

In [ ]:
from common import load_split, label_stats
for s, (nq, ns) in {'train': (1211, 4682), 'val': (256, 988), 'test': (275, 1095)}.items():
    d = load_split(f'{DATA}/{s}.tsv'); st = label_stats(d)
    print(f'{s}: {len(d)} queries, {sum(st.values())} spans ->', 'OK' if (len(d), sum(st.values())) == (nq, ns) else 'DOES NOT MATCH TABLE 2 - STOP')

In [ ]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
SEEDS = [42, 123, 456]          # 3 different sets of 5 examples
for s in SEEDS:
    if os.path.exists(f'{RUNS}/gpt5/seed{s}/metrics.json'):
        print(f'skip gpt5 seed{s} (done)'); continue
    print(f'\n>>> GPT-5.4 five-shot, seed {s}', flush=True)
    !cd /content/code && python gpt_fewshot.py --data_dir "$DATA" --k 5 --seed $s --out_dir "$RUNS/gpt5"

In [ ]:
# Results of the three five-shot runs
import json, glob, sys
import numpy as np
sys.path.insert(0, CODE)
from common import ENTITY_TYPES, read_predictions, per_query_counts, macro_from_totals
f1s = []
for p in sorted(glob.glob(f'{RUNS}/gpt5/seed*/metrics.json')):
    m = json.load(open(p))
    rows = read_predictions(p.replace('metrics.json', 'test_predictions.jsonl'))
    tot = np.array(per_query_counts([r['gold'] for r in rows], [r['pred'] for r in rows])).sum(0)
    P, R, F, per = macro_from_totals(tot[:, 0], tot[:, 1], tot[:, 2]); f1s.append(F)
    print(f"seed {m['seed']}: macro F1 {F:.4f} (P {P:.4f} R {R:.4f}) "
          f"{ {t: round(float(v), 3) for t, v in zip(ENTITY_TYPES, per)} } | parse {m['parse_status']}")
if f1s:
    print(f"\nFive-shot macro F1: {np.mean(f1s):.4f} ± {np.std(f1s, ddof=1) if len(f1s) > 1 else 0:.4f}   (zero-shot in the paper: 0.6682)")